In [ ]:
import os, subprocess, sys, torch

print(f"PyTorch: {torch.__version__}")
print(f"CUDA: {torch.version.cuda}")
print(f"Python: {sys.version}")
print(f"GPU: {torch.cuda.get_device_name(0) if torch.cuda.is_available() else 'None'}")

# Core dependencies
subprocess.run([sys.executable, "-m", "pip", "install", "-q",
    "miditok>=3.0.0",
    "ncps>=0.0.7",
    "safetensors>=0.4.0",
    "pretty_midi>=0.2.10",
    "numpy>=1.24.0",
    "tqdm>=4.65.0",
], check=False)

# Pre-built Mamba wheels for Kaggle (PyTorch 2.9, CUDA 12.x, Python 3.12)
# Update these URLs if Kaggle updates its PyTorch/CUDA version
CAUSAL_CONV1D = (
    "https://github.com/Dao-AILab/causal-conv1d/releases/download/v1.6.1/"
    "causal_conv1d-1.6.1+cu12torch2.9cxx11abiTRUE-cp312-cp312-linux_x86_64.whl"
)
MAMBA_SSM = (
    "https://github.com/state-spaces/mamba/releases/download/v2.3.1/"
    "mamba_ssm-2.3.1+cu12torch2.9cxx11abiTRUE-cp312-cp312-linux_x86_64.whl"
)

if torch.cuda.is_available():
    cc = torch.cuda.get_device_capability(0)
    print(f"GPU compute capability: {cc[0]}.{cc[1]}")
    print("\nInstalling Mamba from pre-built wheels...")
    r1 = subprocess.run([sys.executable, "-m", "pip", "install", "-q",
        CAUSAL_CONV1D], capture_output=True)
    r2 = subprocess.run([sys.executable, "-m", "pip", "install", "-q",
        MAMBA_SSM], capture_output=True)
    if r2.returncode != 0:
        print(f"Mamba wheel failed: {r2.stderr.decode()[-300:]}")
        print("Training will fail unless Mamba is available for this GPU.")
    else:
        print("Mamba installed successfully")
else:
    print("No GPU detected. Mamba requires CUDA and training will not run.")

# Verify mamba
try:
    import mamba_ssm
    print(f"mamba-ssm {mamba_ssm.__version__} active")
except ImportError:
    print("mamba-ssm not available - GRU fallback active")

# Require Mamba by default. To explicitly allow fallback, set:
# os.environ['PIANO_ALLOW_GRU_FALLBACK'] = '1'
os.environ.setdefault('PIANO_REQUIRE_MAMBA', '1')

# Find and add project to Python path
from pathlib import Path

def find_and_add_project():
    marker = 'piano_kaggle_session.py'
    for root, dirs, files in os.walk('/kaggle/input'):
        if marker in files:
            project_root = root
            if project_root not in sys.path:
                sys.path.insert(0, project_root)
            print(f"Project root: {project_root}")
            return project_root
    # Show what's available to help debug
    print("Could not find project. Available in /kaggle/input:")
    for item in Path('/kaggle/input').iterdir():
        print(f"  {item}")
        for subitem in item.iterdir():
            print(f"    {subitem.name}")
    raise FileNotFoundError("piano_kaggle_session.py not found in /kaggle/input")

PROJECT_ROOT = find_and_add_project()
print("Setup complete.")


In [ ]:
# ============================================
# EDIT THIS CELL TO CHANGE TRAINING SETTINGS
# ============================================
SCALE = "nano"        # Options: nano, micro, small, medium
MAX_EPOCHS = 5000     # Ceiling - training stops at plateau before this
# ============================================

from scale_config import list_presets, get_preset
list_presets()
print(f"\nSelected: {SCALE}")
get_preset(SCALE)


In [ ]:
from piano_kaggle_session import run_kaggle_session
# Optional tuning knobs for utilization on multi-GPU runtimes:
# os.environ['PIANO_NUM_WORKERS'] = '8'
# os.environ['PIANO_MAX_BATCH_SIZE'] = '256'
# os.environ['PIANO_GRAD_ACCUM_STEPS'] = '1'
# os.environ['PIANO_DISABLE_AUTO_BATCH'] = '1'  # disable auto scaling

run_kaggle_session(scale=SCALE, max_epochs=MAX_EPOCHS)


In [ ]:
from piano_kaggle_session import calibrate_on_kaggle
calibrate_on_kaggle()


In [ ]:
from piano_kaggle_session import generate_from_seed_file
# Use your generated seed file from /kaggle/working/generated/ or a local path
seed_file = '/kaggle/working/generated/distinct_seed_10s.mid'
generate_from_seed_file(seed_file, scale=SCALE, max_new_tokens=8192)


## Downloading your trained model

After training completes, find your files in the Kaggle output panel:
- `checkpoints/best.safetensors` - best model by validation loss
- `checkpoints/latest.safetensors` - most recent epoch
- `generated/` - sample MIDI continuations generated during training

To use locally:
```bash
python tools/generate_sample.py \
  --checkpoint path/to/best.safetensors \
  --seed your_seed.mid \
  --output generated.mid \
  --scale nano
```

To continue training in a new session:
Re-run the notebook - it auto-resumes from the latest checkpoint
if one exists in /kaggle/working/checkpoints/.


In [ ]:
# Diagnostic cell - run if other cells fail
import os
from pathlib import Path

print("=== Kaggle Environment Diagnostics ===")
print(f"\nWorking dir contents:")
for f in Path('/kaggle/working').iterdir():
    print(f"  {f.name}")

print(f"\n/kaggle/input contents:")
for item in Path('/kaggle/input').iterdir():
    print(f"  {item.name}/")
    try:
        for subitem in list(item.iterdir())[:5]:
            print(f"    {subitem.name}")
    except:
        pass

print(f"\nPython path:")
import sys
for p in sys.path[:5]:
    print(f"  {p}")

print(f"\nInstalled relevant packages:")
import subprocess
result = subprocess.run(['pip', 'show', 'miditok', 'ncps', 'mamba-ssm',
    'safetensors'], capture_output=True, text=True)
print(result.stdout)
